[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/4chan.ipynb)

# 4chan API

Collect data from 4chan through its read-only JSON API: list the boards, read a board's catalog, fetch one complete thread, list the archived threads, and put the pieces together into a collector for a whole board. The sections build on each other and are meant to be run in order.

The API needs no account and no key. The only dependency is `requests`, which
Google Colab has preinstalled.

Rules from the [API documentation](https://github.com/4chan/4chan-API):
at most one request per second, poll a thread no more often than every 10
seconds, and disclose 4chan as the source of anything you publish from it.
Boards can contain offensive and not-safe-for-work content. Read the
[Research considerations](https://yangkclab.github.io/social-media-analysis/topics/data-collection/research-considerations/) page before collecting.

In [ ]:
import requests

## Boards

List every board with `boards.json` and read the per-board settings that matter for data collection.

In [ ]:
boards_url = "https://a.4cdn.org/boards.json"
resp = requests.get(boards_url)
resp.status_code

Output:

```python
200
```

`200` means success. The numbers in the example outputs below are from a run on 2026-08-29; a live run will differ.

In [ ]:
resp_json = resp.json()
boards = resp_json["boards"]
len(boards)

Output:

```python
77
```

In [ ]:
boards[0]

Output:

```python
{'board': '3',
 'title': '3DCG',
 'ws_board': 1,
 'per_page': 15,
 'pages': 10,
 'max_filesize': 4194304,
 'max_webm_filesize': 4194304,
 'max_comment_chars': 2000,
 'max_webm_duration': 120,
 'bump_limit': 310,
 'image_limit': 150,
 'cooldowns': {'threads': 600, 'replies': 60, 'images': 60},
 'meta_description': "&quot;/3/ - 3DCG&quot; is 4chan's board for 3D modeling and imagery.",
 'is_archived': 1}
```

Flags are `1`/`0` integers, not booleans, and a field can be absent instead of `0`: boards without an archive have no `is_archived` key at all, so read it with `.get()`.

The fields you will use:

| Field | Meaning |
|---|---|
| `board` | Short name, used in every other endpoint URL (`/g/`, `/pol/`) |
| `title` | Human-readable name |
| `ws_board` | 1 if the board is work-safe, 0 if not |
| `pages`, `per_page` | How many index pages the board has and how many threads each page holds. Their product is the number of live threads |
| `bump_limit` | After this many replies, new replies stop moving the thread to the top |
| `image_limit` | After this many images, new image posts are refused |
| `is_archived` | 1 if the board keeps an archive of expired threads. The key is absent, not 0, on boards without one, so read it with `.get()`. Boards without an archive cannot be collected through `archive.json` |
| `cooldowns` | Posting cooldowns in seconds. Not relevant for reading |

In [ ]:
for board in boards[:10]:
    print(f"/{board['board']}/  {board['title']}  worksafe={board['ws_board']}  archive={board.get('is_archived', 0)}")

Output:

```text
/3/  3DCG  worksafe=1  archive=1
/a/  Anime & Manga  worksafe=1  archive=1
/aco/  Adult Cartoons  worksafe=0  archive=1
/adv/  Advice  worksafe=1  archive=1
/an/  Animals & Nature  worksafe=1  archive=1
/b/  Random  worksafe=0  archive=0
/bant/  International/Random  worksafe=0  archive=0
/biz/  Business & Finance  worksafe=1  archive=1
/c/  Anime/Cute  worksafe=1  archive=1
/cgl/  Cosplay & EGL  worksafe=1  archive=1
```

In [ ]:
# Boards without an archive
[board["board"] for board in boards if not board.get("is_archived")]

Output:

```python
['b', 'bant', 'f', 'trash']
```

In [ ]:
# Work-safe boards versus the rest
sum(board["ws_board"] for board in boards), len(boards)

Output:

```python
(53, 77)
```

53 of the 77 boards were marked work-safe on 2026-08-29.

## Catalog

Read every live thread on a board with `catalog.json`. The catalog is the board's front pages as JSON: one entry per page, each holding the original post (OP) of every thread on that page and a preview of its last replies.

In [ ]:
catalog_url = "https://a.4cdn.org/{board}/catalog.json"
resp = requests.get(catalog_url.format(board="g"))
catalog = resp.json()
len(catalog)

Output:

```python
11
```

The catalog is a list of pages — 11 of them for `/g/` that day.

In [ ]:
page_1 = catalog[0]
page_1.keys()

Output:

```python
dict_keys(['page', 'threads'])
```

In [ ]:
len(page_1["threads"])

Output:

```python
15
```

Each entry in `threads` is the OP of one thread plus a few summary fields. The first thread on page 1 of `/g/` is the board's sticky, which is why `sticky` and `closed` are set here.

In [ ]:
thread = page_1["threads"][0]
summary_fields = ["no", "resto", "sticky", "closed", "time", "replies", "images", "last_modified"]
{field: thread.get(field) for field in summary_fields}

Output:

```python
{'no': 105076684,
 'resto': 0,
 'sticky': 1,
 'closed': 1,
 'time': 1745612650,
 'replies': 3,
 'images': 3,
 'last_modified': 1745612922}
```

This is the `/g/` sticky thread (`sticky: 1`, `closed: 1`), pinned at the top of page 1 since April 2025 — which makes it a stable example.

The fields you will use:

| Field | Meaning |
|---|---|
| `no` | Post number. For the OP this is also the thread ID |
| `resto` | The thread the post belongs to. 0 for an OP |
| `time` | Unix timestamp of the post |
| `now` | The same time as a string in US Eastern time. Use `time` |
| `sub`, `com` | Subject and body. `com` is HTML, not plain text |
| `filename`, `ext`, `tim` | The attached image. The file URL is `https://i.4cdn.org/{board}/{tim}{ext}` |
| `replies`, `images` | Counts for the whole thread |
| `last_replies` | At most the last 5 replies, not all of them |
| `last_modified` | Unix timestamp of the last change to the thread |

In [ ]:
# last_replies is a preview: at most five replies, so the catalog alone
# never gives you a complete thread
busy = page_1["threads"][1]
busy["replies"], [(reply["no"], reply["time"]) for reply in busy["last_replies"]]

Output:

```python
(159,
 [(109682328, 1788035889),
  (109682591, 1788038103),
  (109682621, 1788038448),
  (109682647, 1788038637),
  (109682659, 1788038686)])
```

159 replies in the thread, but `last_replies` carries only the newest 5.

In [ ]:
# All live thread IDs on the board, across every page
thread_ids = [thread["no"] for page in catalog for thread in page["threads"]]
len(thread_ids), thread_ids[:5]

Output:

```python
(151, [105076684, 109660156, 109646078, 109681974, 109680642])
```

151 live threads across the 11 pages.

## Thread

Read one complete thread, the OP and every reply, with `/{board}/thread/{op_id}.json`. This is the only endpoint that returns all replies. The example uses the `/g/` sticky thread, which is written by moderators and never expires.

In [ ]:
thread_url = "https://a.4cdn.org/{board}/thread/{op_id}.json"
resp = requests.get(thread_url.format(board="g", op_id=105076684))
resp.status_code

Output:

```python
200
```

A `404` means the thread no longer exists. Threads are deleted some time after they are archived, so check the status code before calling `.json()`.

In [ ]:
posts = resp.json()["posts"]
len(posts)

Output:

```python
4
```

The first post is the OP; the rest are replies, in posting order.

In [ ]:
posts[0]

Output, with the `com` HTML shortened:

```python
{'no': 105076684,
 'sticky': 1,
 'closed': 1,
 'now': '04/25/25(Fri)16:24:10',
 'name': 'Anonymous',
 'com': 'This board is for the discussion of technology and related topics.<br>\r\n<br>\r\nReminder that instigating OR participating in flame/brand wars will result in a ban.<br>\r\nTech support threads should be posted to <a href="https://boards.4chan.org/wsr/">...</a> ...',
 'filename': 'sticky btfo',
 'ext': '.png',
 'w': 535,
 'h': 420,
 'tn_w': 250,
 'tn_h': 196,
 'tim': 1745612650141704,
 'time': 1745612650,
 'md5': 'zuZHMJMYYp5WY7vM397nWQ==',
 'fsize': 301273,
 'resto': 0,
 'capcode': 'mod',
 'semantic_url': 'this-board-is-for-the-discussion-of-technology',
 'replies': 3,
 'images': 3}
```

`com` is HTML, not plain text: line breaks are `<br>`, quotes are `<span class="quote">`, and `>>` reply links arrive HTML-escaped inside `<a class="quotelink">` tags. `time` is a Unix timestamp in seconds; `tim` is the image upload timestamp in microseconds and names the image file.

The fields you will use:

| Field | Meaning |
|---|---|
| `no` | Post number |
| `resto` | Thread ID. 0 for the OP, the OP's `no` for every reply |
| `time` | Unix timestamp |
| `name` | Poster name. Almost always `Anonymous` |
| `trip`, `id` | Tripcode and poster ID. Only present on some boards and posts |
| `com` | Post body as HTML. Quote links look like `<a href="#p123" class="quotelink">&gt;&gt;123</a>` |
| `filename`, `ext`, `tim`, `w`, `h`, `md5` | The attached image, if any |
| `replies`, `images` | OP only. Counts for the thread |
| `archived`, `archived_on` | OP only. Set once the thread has expired |

In [ ]:
for post in posts:
    print(post["no"], post["resto"], post["time"], post.get("ext"))

Output:

```text
105076684 0 1745612650 .png
105076685 105076684 1745612666 .png
105076689 105076684 1745612673 .gif
105076692 105076684 1745612680 .png
```

`resto` is `0` on the OP and the OP's `no` on every reply.

### Who replies to whom

4chan has no reply button. A reply quotes another post by writing `>>123` in
its body, which the site renders as a link. In the JSON the body is HTML, so
the two `>` characters arrive as `&gt;&gt;`. A regular expression recovers the
post numbers, and from them you can build the reply tree of a thread.

The sticky has no quote links, so the cell below picks the busiest thread on
page 1 of the catalog and prints post numbers only.

In [ ]:
import re

def quoted_posts(post):
    """Post numbers that this post replies to."""
    return [int(number) for number in re.findall(r"&gt;&gt;(\d+)", post.get("com", ""))]

In [ ]:
catalog = requests.get("https://a.4cdn.org/g/catalog.json").json()
live = [thread for thread in catalog[0]["threads"] if not thread.get("sticky")]
busiest = max(live, key=lambda thread: thread["replies"])
resp = requests.get(thread_url.format(board="g", op_id=busiest["no"]))
posts = resp.json()["posts"]
len(posts)

Output:

```python
279
```

In [ ]:
for post in posts[:12]:
    print(post["no"], "->", quoted_posts(post))

Output:

```text
109664794 -> []
109664804 -> [109664794]
109664809 -> [109664804]
109664814 -> [109664809]
109664823 -> [109664814]
109664828 -> [109664823]
109664887 -> [109664828, 109664823]
109664901 -> [109664887]
109664906 -> [109664794]
109664976 -> [109664906]
109665010 -> [109664794]
109665177 -> [109664794, 109664976]
```

An empty list means the post replies to the thread as a whole. `>>` links can also point to other threads or other boards, so a quoted number is not always in the same thread.

## Archive

List the threads that have expired on a board with `archive.json`. A thread expires when it is pushed off the board's last page. It is then read-only, stays available for a while, and is deleted. The archive is a plain list of thread IDs, most recent last.

In [ ]:
archive_url = "https://a.4cdn.org/{board}/archive.json"
resp = requests.get(archive_url.format(board="g"))
archived = resp.json()
len(archived)

Output:

```python
1462
```

1,462 archived threads on `/g/` that day, as a flat list of thread IDs.

In [ ]:
archived[:5], archived[-5:]

Output:

```python
([109444547, 109495988, 109497359, 109504557, 109521167],
 [109677736, 109677783, 109678036, 109678268, 109678926])
```

The list is ordered oldest first, so the last entries are the threads archived most recently.

Not every board has an archive. `boards.json` reports it in `is_archived`; at the time of writing `/b/`, `/bant/`, `/f/`, and `/trash/` have none. An archived thread is still readable through the thread endpoint until it is deleted, and its OP carries `archived` and `archived_on`.

In [ ]:
thread_url = "https://a.4cdn.org/{board}/thread/{op_id}.json"
resp = requests.get(thread_url.format(board="g", op_id=archived[-1]))
op = resp.json()["posts"][0]
{field: op.get(field) for field in ["no", "time", "replies", "images", "archived", "archived_on"]}

Output:

```python
{'no': 109678926,
 'time': 1788012460,
 'replies': 3,
 'images': 0,
 'archived': 1,
 'archived_on': 1788036212}
```

`archived_on` minus `time` is the thread's lifetime — here about 6.6 hours from posting to archiving.

Why this matters for collection: an archived thread does not change any more, so fetching it once gives you the complete thread. Polling the archive, and fetching every ID you have not seen before, is the simplest way to collect a whole board. The next notebook does exactly that.

## Collect a whole board

Collect every thread on a board, with all of its replies, by polling the archive. This section runs one round. In a real collector the same round runs from `cron` every few minutes for as long as the study lasts.

### Why the archive

Two ways to get every thread on a board:

1. **Poll the catalog.** Easy, but `last_replies` holds only five replies, so
   you never get complete threads, and a thread that appears and expires
   between two polls is missed.
2. **Poll the archive.** An archived thread is complete and no longer changes.
   Fetch each newly archived thread once, before it is deleted, and you have
   the whole board. The cost is a delay: you see a thread only after it expires.

Approach 2 needs three things: a record of the thread IDs already fetched, a
polling interval shorter than the time between archiving and deletion, and a
scheduler that keeps running when you are not looking.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

In [ ]:
BOARD = "g"
OUT_FILE = Path(f"4chan_{BOARD}_threads.jsonl")   # one thread per line
SEEN_FILE = Path(f"4chan_{BOARD}_seen.json")      # thread IDs already fetched
MAX_PER_ROUND = 3                                  # small for the demo; drop the cap in production
HEADERS = {"User-Agent": "cs415-course-demo"}

In [ ]:
def load_seen():
    if SEEN_FILE.exists():
        return set(json.loads(SEEN_FILE.read_text()))
    return set()

def save_seen(seen):
    SEEN_FILE.write_text(json.dumps(sorted(seen)))

seen = load_seen()
len(seen)

Output:

```python
0
```

On the first run the seen-set file does not exist yet, so nothing has been seen.

In [ ]:
def get_archive(board):
    resp = requests.get(f"https://a.4cdn.org/{board}/archive.json", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def get_thread(board, op_id):
    resp = requests.get(f"https://a.4cdn.org/{board}/thread/{op_id}.json", headers=HEADERS)
    if resp.status_code == 404:
        return None          # deleted between the archive poll and now
    resp.raise_for_status()
    return resp.json()["posts"]

In [ ]:
# One round: find newly archived threads and fetch them
archived = get_archive(BOARD)
new_ids = [op_id for op_id in archived if op_id not in seen]
len(archived), len(new_ids)

Output:

```python
(1462, 1462)
```

On the first round every archived thread is new. On later rounds the first number stays around the board's archive size and the second is only the handful archived since the last run.

In [ ]:
fetched, missing = 0, 0
with OUT_FILE.open("a") as out:
    for op_id in new_ids[:MAX_PER_ROUND]:
        posts = get_thread(BOARD, op_id)
        time.sleep(1)                               # API rule: at most one request per second
        seen.add(op_id)                             # do not retry a deleted thread either
        if posts is None:
            missing += 1
            continue
        record = {
            "board": BOARD,
            "thread_id": op_id,
            "collected_at": datetime.now(timezone.utc).isoformat(),
            "posts": posts,                         # the raw response, untouched
        }
        out.write(json.dumps(record) + "\n")
        fetched += 1
save_seen(seen)
fetched, missing, len(seen)

Output:

```python
(3, 0, 3)
```

3 threads fetched, 0 already deleted, 3 IDs now in the seen-set — the demo caps a round at `MAX_PER_ROUND = 3` so it finishes in seconds. A real collector would not cap it.

In [ ]:
# What was written
with OUT_FILE.open() as f:
    records = [json.loads(line) for line in f]
len(records), [(r["thread_id"], len(r["posts"])) for r in records]

Output:

```python
(3, [(109444547, 315), (109495988, 321), (109497359, 310)])
```

One JSONL record per thread, each carrying the full post list — 315, 321, and 310 posts here.

### Running it for real

- **Schedule it.** Save the cells above as a script and run it from `cron`.
  This line runs it every 10 minutes:

  ```
  */10 * * * * /path/to/python /path/to/collect_4chan.py >> /path/to/collect.log 2>&1
  ```

  The 2022 U.S. midterm dataset used the same design: the catalog every 5
  minutes and the archive every 10.
- **Pick the interval from the data.** Compare two archive polls a few hours
  apart. If IDs disappear from the front of the list faster than you poll,
  you are losing threads.
- **Keep `seen` on disk**, as above. A restart must not refetch everything,
  and it must not lose the IDs it already has.
- **Store the raw response.** Derive fields later. Record when you collected
  each thread, not only when it was posted.
- **Remove `MAX_PER_ROUND`.** The cap is only there so the demo finishes in
  seconds. The first real round fetches the whole archive, about 1,500 threads
  on `/g/`, which takes about 25 minutes at one request per second.
- **Log every round.** A collector that dies overnight and nobody notices is
  the most common failure in Project 1.
- **Images are separate.** The posts carry file names, not files. Download
  images only if your research question needs them, and check the topic page's
  research considerations first.